In [ ]:
import os, sys
import math, time
import numpy as np
import pandas as pd
import collections
import random

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.data import RandomSampler, SequentialSampler

from collections import defaultdict
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
!pip uninstall transformers -y
!pip install transformers==4.40.2

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

from transformers import BertTokenizer
from transformers import BertForSequenceClassification
from transformers import get_scheduler
import torch.optim as AdamW
# from transformers import AdamW

In [ ]:
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

In [ ]:
# Reference: https://github.com/huggingface/transformers/
# Config File: /src/transformers/models/bert/configuration_bert.py

# V = Vocabulary size
# H: Hidden size (or d_model)
# L: Number of hidden Layers
# A: Number of Attention heads
# I: Intermediate size (feed-forward layer)
# S: Max Sequence length (max position embeddings)

V = 30522
H = 768
L = 2
A = 12
I = 768 * 4
S = 512

# T: Token Type vocabulary size
# p: Dropout probability
# C: Number of output Classes (labels)

T = 2
p = 0.1
C = 2

**1. embTokens (Token Embeddings)** : Maps each token ID from the vocabulary (size V) to a learned vector (size H).

**2. embPositional (Positional Embeddings)** : Maps each position index (from 0 to S - 1) to a unique vector (size H).

**3. embTokenType (Token Type  Embeddings)** : Maps a segment ID (0 for Sentence A, 1 for Sentence B) to a vector (size H).


---
**How They Combine:**
The final embedding E for a token is the **sum** of these three vectors.

E = embTokens + embPositional + embTokenType

In [ ]:
class BERTEmbedder(nn.Module):
    def __init__(self):
        super().__init__()
        self.embTokens = nn.Embedding(V, H)      # Reduce V to H
        self.embPositional = nn.Embedding(S, H)  # Reduce S to H
        self.embTokenType = nn.Embedding(T, H)   # Reduce T to H

        self.normalizer = nn.LayerNorm(H)
        self.dropout = nn.Dropout(p)

    def forward(self, inputIDs, tokTypeIDs=None):
        len = inputIDs.size(1)
        posnIDs = torch.arange(len, dtype = torch.long, device = inputIDs.device)
        posnIDs = posnIDs.unsqueeze(0) # (1, len)

        if tokTypeIDs is None: tokTypeIDs = torch.zeros_like(inputIDs)

        emb1 = self.embTokens(inputIDs)
        emb2 = self.embPositional(posnIDs)
        emb3 = self.embTokenType(tokTypeIDs)
        # print(emb1.shape)
        # print(emb2.shape)
        # print(emb3.shape)

        E = emb1 + emb2 + emb3
        E = self.normalizer(E)
        E = self.dropout(E)
        # print(E.shape)
        return E

In [ ]:
embedder = BERTEmbedder()
embedder.eval()

# Consider BATCH_SIZE = 2, SEQ_LEN = 10

inputIDs = torch.randint(0, V, (2, 10))
tokTypeIDs = torch.zeros((2, 10), dtype = torch.long)
hidden = torch.rand(2, 10, H)

output = embedder(inputIDs, tokTypeIDs)
print(inputIDs.shape)
print(tokTypeIDs.shape)
print(output.shape)

In [ ]:
# Prepare a tensor (Query, Key, or Value) for multi-head attention:

def getTensors(x):
    X = x.size()[:-1]
    headSize = H // A
    X += (A, H // A)
    # print(X)

    x = x.view(*X)
    x = x.permute(0, 2, 1, 3)
    # print(x.shape) #(B, A, S, h)
    return x

def getVector(x):
    B, A, S, headSize = x.size()
    x = x.permute(0, 2, 1, 3).contiguous()
    X = x.view(B, S, A * headSize)
    # print(X.shape)
    return X

T1 = torch.rand(2, 10, H)
T2 = getTensors(T1)
T3 = getVector(T2)

print()
print(T1.shape)
print(T2.shape)
print(T3.shape)
# print(T1 == T3) # Verify

In [ ]:
class MultiHeadSelfAttention(nn.Module):
  def __init__(self):
    super().__init__()
    self.heads = A
    self.hiddenDim = H
    self.headSize = H // A

    self.query = nn.Linear(H, H) # H x H size for Q
    self.key   = nn.Linear(H, H) # H x H size for K
    self.value = nn.Linear(H, H) # H x H size for V

    self.dropout = nn.Dropout(p)
    self.FC = nn.Linear(H, H)

  def forward(self, h):
    QV = self.query(h)
    KV = self.key(h)
    VV = self.value(h)

    QT = getTensors(QV)
    KT = getTensors(KV)
    VT = getTensors(VV)

    # Scaled dot-product Attention Scores
    # S = (QT . KT) / sqrt(H)
    # H = size of one head

    score = torch.matmul(QT, KT.transpose(-1, -2))
    score = score / math.sqrt(self.headSize)
    # print(score.shape)

    probs = nn.Softmax(dim = -1)(score) # normalising scores
    AM = self.dropout(probs) # AM = attention matrix
    # print(AM.shape)

    output = torch.matmul(AM, VT)
    output = getVector(output)
    # print(output.shape)
    output = self.FC(output)
    return output

In [ ]:
AttentionLayer = MultiHeadSelfAttention()
AttentionLayer.eval()

o1 = AttentionLayer(hidden)
print({hidden.shape})
print({o1.shape})

In [ ]:
class FeedForwardLayer(nn.Module):
  def __init__(self):
    super().__init__()
    self.dense1 = nn.Linear(H, I) # I = 4 X H
    self.activation = nn.GELU() # using GELU
    self.dense2 = nn.Linear(I, H)
    self.dropout = nn.Dropout(p)

  def forward(self, h):
    f1 = self.dense1(h)
    a1 = self.activation(f1)
    # print(a1.shape)
    f2 = self.dense2(a1)
    output = self.dropout(f2)
    return output

In [ ]:
FFLayer = FeedForwardLayer()
FFLayer.eval()

o2 = FFLayer(o1)
print({o1.shape})
print({o2.shape})

In [ ]:
class Encoder(nn.Module):
  def __init__(self):
    super().__init__()
    self.attention = MultiHeadSelfAttention()
    self.FFN = FeedForwardLayer()
    self.LN1 = nn.LayerNorm(H)
    self.LN2 = nn.LayerNorm(H)
    self.dropout = nn.Dropout(p)


  def forward(self, h):
    # MHSA -> (Add + LN) -> FFN -> (Add + LN)
    a1 = self.attention(h)
    a1 = self.dropout(a1)

    res1 = self.LN1(h + a1)
    ffn = self.FFN(res1)
    res2 = self.LN2(res1 + ffn)
    return res2

In [ ]:
# Reference Used: src/transformers/models/bert/modeling_bert.py#L691
class BertPooler(nn.Module):
    def __init__(self):
        super().__init__()
        self.dense = nn.Linear(H, H)
        self.activation = nn.Tanh()

    def forward(self, h):
        CLS = h[:, 0] # CLS = tensor for [CLS] token
        output = self.dense(CLS)
        output = self.activation(output)
        return output

In [ ]:
class MyBERTModel(nn.Module):
  def __init__(self):
    super().__init__()
    self.embeddings = BERTEmbedder()

    # L = 2 : we need to stack to encoder layers
    self.encoder1 = Encoder()
    self.encoder2 = Encoder()
    self.pooler = BertPooler()
    self.classifier = nn.Linear(H, C)

  def forward(self, inputIDs, tokTypeIds = None):
    emb = self.embeddings(inputIDs, tokTypeIds)
    enc1 = self.encoder1(emb)
    enc2 = self.encoder2(enc1)
    # print(emb.shape)
    # print(enc1.shape)
    # print(enc2.shape)

    pool = self.pooler(enc2)
    logits = self.classifier(pool)
    # print(pooler.shape)
    # print(logits.shape)
    return logits

In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased', use_fast=False)
test = "Hey, I'm Tuhin. Check out my Github Profile @Ecolash! hehe :) "

inputs = tokenizer(test, return_tensors = "pt", padding = True, truncation = True, max_length = S)
print(type(inputs))
print(list(inputs.keys()))
print()

inputIDs = inputs['input_ids']
tokTypeIDs = inputs['token_type_ids']
mask = inputs['attention_mask']

# print(inputIDs)
# print(tokTypeIDs)
# print(mask)

print(inputIDs.shape)
print(tokTypeIDs.shape)
print(mask.shape)

In [ ]:
model = MyBERTModel()
model.eval()

logits = model(inputIDs, tokTypeIDs)
probs = nn.Softmax(dim = -1)(logits)
label = torch.argmax(probs, dim = -1)

print(logits)
print(probs)
print(label)

In [ ]:
DATA_PATH = '/kaggle/input/fakeddit-nlp-assignment'
cols = ['id', 'clean_title', '2_way_label']

file1 = 'all_train.tsv'
path1 = os.path.join(DATA_PATH, file1)
print(f"[+] Loading {path1}\n")

df1 = pd.read_csv(path1, sep='\t')
[print(f" [+] Required col found: {c}") for c in cols if c in df1.columns];
print(f" [+] Loaded {len(df1)} rows.")
df1 = df1[cols]

In [ ]:
file2 = 'all_validate.tsv'
path2 = os.path.join(DATA_PATH, file2)
print(f"[+] Loading {path2}\n")

df2 = pd.read_csv(path2, sep='\t')
[print(f" [+] Required col found: {c}") for c in cols if c in df2.columns];
print(f" [+] Loaded {len(df2)} rows.")
df2 = df2[cols]

In [ ]:
file3 = 'all_test_public.tsv'
path3 = os.path.join(DATA_PATH, file3)
print(f"[+] Loading {path3}\n")

df3 = pd.read_csv(path3, sep='\t')
[print(f" [+] Required col found: {c}") for c in cols if c in df3.columns];
print(f" [+] Loaded {len(df3)} rows.")
df3 = df3[cols]

In [ ]:
dfs = [df1, df2, df3]
for i, d in enumerate(dfs, 1): print(f"[+] df{i} shape: {d.shape}")

combined = pd.concat(dfs, ignore_index=True)
print(f"\n[+] Combined shape (before cleaning): {combined.shape}")

NAs = combined.isna().sum()
totalNAs = int(NAs.sum())
print(f"[+] NaN counts: {totalNAs}\n")

if totalNAs > 0:
    before = combined.shape[0]
    combined = combined.dropna()
    removed = before - combined.shape[0]
    print(f"[+] Dropped {removed} rows containing NaNs")
    print(f"[+] Combined shape (after cleaning): {combined.shape}")

In [ ]:
print("\n[+] Statistics:")
print(combined.info())
print(combined.head(1))

In [ ]:
COMMENTS_PATH = '/kaggle/input/fakeddit-nlp-assignment/all_comments.tsv'
cols = ['id', 'submission_id', 'body', 'parent_id']

file = 'all_comments.tsv'
path = os.path.join(COMMENTS_PATH, file)
print(f"[+] Loading {path}\n")

cols = ["id", "submission_id", "body", "parent_id"]
comments = pd.read_csv(
    path,
    sep = "\t",
    usecols = lambda c: c in cols,
    dtype = str,
    # low_memory = False,
    on_bad_lines = "skip",
    engine="python"    
)

[print(f" [+] Required col found: {c}") for c in cols if c in comments.columns];
print(f" [+] Loaded {len(comments)} comments.")

comments = comments.dropna(subset=["submission_id", "body", "parent_id"])
comments["submission_id"] = comments["submission_id"].astype(str)
print(f" [+] Filtered {len(comments)} comments.")

In [ ]:
comment_T1 = comments[comments["parent_id"].str.startswith("t3_")].copy() # Direct comments
comment_T2 = comments[comments["parent_id"].str.startswith("t1_")].copy() # Reply to comments

# frac = 0.005
# comment_T1 = comment_T1.sample(frac = frac, random_state = 42)
# comment_T2 = comment_T2.sample(frac = frac, random_state = 42)

print(f"[+] Top level comments: {len(comment_T1)}")
print(f"[+] Replies to comments: {len(comment_T2)}")

# print(comment_T1.info())

In [ ]:
def buildThread(parentID, depth=1, limit=3, prefix=""):
    key = 'NOT AVAILABLE'
    if parentID.startswith("t1_"): key = parentID     
    if parentID.startswith("t3_"): key = parentID    
    if key not in children: return ""
    if depth >= limit: return ""
        
    replies = []
    for i, (replyID, text) in enumerate(children[key], start=1):
        newPrefix = f"{prefix}{i}" if prefix == "" else f"{prefix}.{i}"
        tag = f"[REPLY {newPrefix}] {text}"
        replies.append(tag)

        newParentID = "t1_" + replyID
        subReplies = buildThread(newParentID, depth + 1, limit, newPrefix)
        if subReplies: replies.append(subReplies)

    return "\n".join(replies).strip()

commentID = "t3_c0"
comment = "This is the original comment."

children = defaultdict(list)
children["t3_c0"] = [("c1", "Reply 1"), ("c2", "Reply 2")]
children["t1_c1"] = [("c3", "Reply to Reply 1")]

text = f"[COMMENT] {comment} \n" + buildThread(commentID)
print(text)

In [ ]:
def buildThreadv2(parentID, links, depth = 1,prefix = ""):
    thread = []
    children = links.get(parentID, [])
    
    for i, child in enumerate(children):
        newPrefix = f"{prefix}.{i+1}" if prefix else f"{i+1}"
        reply = f"[REPLY {newPrefix}] {child['body']}"
        thread.append(reply)
        
        subReplies = buildThreadv2(child['id'], links, depth, newPrefix)
        thread.extend(subReplies)
        
    return thread

In [ ]:
from tqdm.notebook import tqdm
print("[+] Pre-processing replies")
comment_T2['parID'] = comment_T2['parent_id'].str.slice(3)

links = {}
total = comment_T2['parID'].nunique()
for id, group in tqdm(comment_T2.groupby('parID'), total = total, desc = "[+] Building a connection MAP: "):
    rows = group[['id', 'body']]
    records = rows.to_dict('records')
    links[id] = records

# print(f"[+] Building {len(comment_T1)} threads...")

results = []
for comment in tqdm(comment_T1[['id', 'body', 'parent_id']].itertuples(index=False), total=len(comment_T1), desc = "[+] Building threads:"):
    t1_text = f"[COMMENT] {comment.body}"
    reply = buildThreadv2(comment.id, links)

    postID = comment.parent_id[3:]
    text = "\n".join([t1_text] + reply)
    results.append({'postID': postID,'text': text})

print("[+] Thread building complete.")
print("[+] Creating Thread DF")

threadsDF = pd.DataFrame(results)

In [ ]:
print("\n[+] Statistics:")
print(threadsDF.info())
print(threadsDF.head(1))

In [ ]:
print(f"[+] Loaded combined DF: {len(combined)} rows")
print(f"[+] Loaded threads DF: {len(threadsDF)} rows")
print(f"[+] Collecting comments for same post (Group by PostID)...")

grouped = threadsDF.groupby('postID')['text'] # group comments under same post
allComments = grouped.apply('\n'.join) 
threadsDF = allComments.reset_index()

print(f"[+] Aggregated threads DF: {len(threadsDF)} rows")

mergedDF = pd.merge(
    combined,
    threadsDF,
    left_on='id',
    right_on='postID',
    how='left'
)

print(f"[+] Merged both DFs: {len(mergedDF)} rows")
mergedDF['text'] = mergedDF['text'].fillna('')
mergedDF['text'] = '[POST] ' + mergedDF['clean_title'] + '\n' + mergedDF['text']
mergedDF['text'] = mergedDF['text'].str.rstrip('\n')

finalDF = mergedDF[['id', 'text', '2_way_label']]
finalDF = finalDF.rename(columns={'2_way_label': 'label'})
print(f"[+] Created final DF: {len(mergedDF)} rows")

In [ ]:
check = finalDF['text'].str.contains('[COMMENT]', regex=False)
n1 = check.sum()
n2 = len(finalDF) - n1

print(f"[+] Total rows in finalDF: {len(finalDF)}")
print(f"[+] Rows with comments: {n1}")
print(f"[+] Rows without comments: {n2}")
print("\n[+] Statistics:")
print(finalDF.info())
print(finalDF.head(1))

In [ ]:
outputFile = 'fakeddit-final.csv'
print(f"[+] Saving finalDF to {outputFile}...")

finalDF.to_csv(outputFile, index=False)
print(f"[+] Successfully saved {len(finalDF)} rows.")

In [ ]:
FRACTION = 0.1
MIN_SAMPLES = 2500

# 2-way labels: 
fakeDF = finalDF[finalDF['label'] == 1].copy() # 1 = Fake [-]
realDF = finalDF[finalDF['label'] == 0].copy() # 0 = Real [+]

print(f"[+] Fake Posts DF: {fakeDF.shape}")
print(f"[+] Real Posts DF: {realDF.shape}")
print(f"[+] Original data counts: {len(fakeDF)} Fake, {len(realDF)} Real\n")

fakeCount = len(fakeDF)
realCount = len(realDF)

n1 = int(fakeCount * FRACTION)
n2 = int(realCount * FRACTION)
n1 = max(MIN_SAMPLES, n1)
n2 = max(MIN_SAMPLES, n2)

print(f"[+] Sampling {n1} posts for fake class.")
print(f"[+] Sampling {n2} posts for real class.")

df_fake = fakeDF.sample(n1, random_state = 42)
df_real = realDF.sample(n2, random_state = 42)
DF = pd.concat([df_fake, df_real])
DF = DF.sample(frac = 1, random_state = 42).reset_index(drop=True) #shuffle for fun
print(f"[+] Total posts in balanced dataset: {len(DF)}")

In [ ]:
# Train-Test Split 
# stratify => ensures balanced labels (0, 1)
X, y = train_test_split(DF, test_size = 0.2,random_state = 42, stratify = DF['label'])

print(f"[+] Train set size: {len(X)}")
print(f"[+] Test set size:  {len(y)}")

In [ ]:
print("[>] Statistics Report")
print(f"\n[+] Number of posts in original dataset: {len(finalDF)}")

counts = DF['label'].value_counts()
print("[+] Number of posts in balanced dataset (DF):")
print(f"   [+] Real (0): {counts.get(0, 0)}")
print(f"   [-] Fake (1): {counts.get(1, 0)}")
print(f"   [=] Total :{len(DF)}")

print("\n[+] Distribution in train/test sets:")
print(f"   [>] Train set (X): {len(X)} posts")
print(f"   [>] Test set (y):  {len(y)} posts")

print("\n[+] Comment Statistics:")
print(f"   [>] Total comments loaded (comments DF): {len(comments)}")
print(f"   [>] Total direct comments (comment_T1):  {len(comment_T1)}")
print(f"   [>] Total reply comments (comment_T2):   {len(comment_T2)}")
    
finalDF['num_comments'] = finalDF['text'].str.count(r'\[COMMENT\]')
findComments = (finalDF['num_comments'] > 0).sum()
print(f"\n   [+] Posts with at least one comment (original): {findComments}")

statsFake = finalDF[finalDF['label'] == 1]['num_comments'].agg(['mean', 'std'])
statsReal = finalDF[finalDF['label'] == 0]['num_comments'].agg(['mean', 'std'])

print("   [+] Mean + Standard Deviation of comments (Original Dataset):")
print(f"       [>] Fake Posts: {statsFake['mean']:.2f} mean / {statsFake['std']:.2f} std")
print(f"       [>] Real Posts: {statsReal['mean']:.2f} mean / {statsReal['std']:.2f} std")
    
DF['num_comments'] = DF['text'].str.count(r'\[COMMENT\]')
statsFake = DF[DF['label'] == 1]['num_comments'].agg(['mean', 'std'])
statsReal = DF[DF['label'] == 0]['num_comments'].agg(['mean', 'std'])

print("\n   [+] Mean + Standard Deviation of comments (Balanced Dataset):")
print(f"       [>] Fake Posts: {statsFake['mean']:.2f} mean / {statsFake['std']:.2f} std")
print(f"       [>] Real Posts: {statsReal['mean']:.2f} mean / {statsReal['std']:.2f} std")

In [ ]:
class FakedditDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        print(f"[+] Initialized FakedditDataset: {len(texts)} samples, MAX LEN = {max_len}")

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = str(self.texts[item])
        label = self.labels[item]
        
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False, # We don't need segment IDs
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        
        return {
            'text': text,
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased', use_fast=False)
print(f"[+] Loaded bert-base-uncased tokenizer for tokenization.")

train_texts = X['text'].values
train_labels = X['label'].values

test_texts = y['text'].values
test_labels = y['label'].values

MAX_LEN = 256 # = S / 2: Max Sequence length (max position embeddings)
BATCH_SIZE = 16

train_dataset = FakedditDataset(train_texts, train_labels, tokenizer, MAX_LEN)
test_dataset = FakedditDataset(test_texts, test_labels, tokenizer, MAX_LEN)

train_loader = DataLoader(
    train_dataset,
    sampler = RandomSampler(train_dataset),
    batch_size = BATCH_SIZE,
    drop_last = True,  
    pin_memory = True,
    num_workers = 2
)

test_loader = DataLoader(
    test_dataset,
    sampler = SequentialSampler(test_dataset),
    batch_size = BATCH_SIZE,
    pin_memory = True,
    num_workers = 2
)

print(f"\n[+] Created train_loader with {len(train_loader)} batches.")
print(f"[+] Created test_loader with {len(test_loader)} batches.")

In [ ]:
print("\n[>] Testing DataLoaders by fetching one batch...")

train_batch = next(iter(train_loader))
print("\n[+] Train Loader Test Batch:")
print(f"    [>] Keys: {train_batch.keys()}")
print(f"    [>] input_ids shape: {train_batch['input_ids'].shape}")
print(f"    [>] attn_mask shape: {train_batch['attention_mask'].shape}")
print(f"    [>] O/P label shape: {train_batch['labels'].shape}")
print(f"    [>] Sample labels: {train_batch['labels'][:5]}")

test_batch = next(iter(test_loader))
print("\n[+] Test Loader Test Batch:")
print(f"    [>] Keys: {test_batch.keys()}")
print(f"    [>] input_ids shape: {test_batch['input_ids'].shape}")
print(f"    [>] attn_mask shape: {test_batch['attention_mask'].shape}")
print(f"    [>] O/P label shape: {test_batch['labels'].shape}")
print(f"    [>] Sample labels: {test_batch['labels'][:5]}")



In [ ]:
EPOCHS = 3
LEARNING_RATE = 2e-5 
OPTIMIZER = 'AdamW'
MODEL_NAME = 'bert-base-uncased'

print("[>] Load Model and Optimizer")

model = BertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels = C, # Fake (1) or Non-Fake (0)
    output_attentions = False,
    output_hidden_states = False
)
model = model.to(device)
print(f"[+] Loaded 'my-bert-model' to {device}.")

from torch.optim import AdamW
print("[>] Defining Optimizer...")
optimizer = AdamW(model.parameters(), lr = LEARNING_RATE, eps = 1e-9)

from transformers import get_linear_schedule_with_warmup
print("[>] Defining Scheduler...")
steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps = 1, 
    num_training_steps = steps
)

print("[+] Optimizer and scheduler configured.")

print("\n[!] Hyperparameters Used for Training:")
print(f"[!]   MAX_LEN:       {MAX_LEN}")
print(f"[!]   BATCH_SIZE:    {BATCH_SIZE}")
print(f"[!]   EPOCHS:        {EPOCHS}")
print(f"[!]   LEARNING_RATE: {LEARNING_RATE}")
print(f"[!]   MODEL_NAME:    {MODEL_NAME}")
print(f"[!]   OPTIMIZER:     {OPTIMIZER}")

In [ ]:
print("[>] Starting Task: Training Loop")

def train_epoch(model, data_loader, optimizer, device, scheduler):
    model.train() 
    
    total_loss = 0
    total_correct = 0
    total_samples = 0
    
    start_time = time.time()
    
    for i, batch in enumerate(data_loader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        model.zero_grad()
        
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        
        loss = outputs.loss
        logits = outputs.logits
        
        total_loss += loss.item()
        
        _, preds = torch.max(logits, dim=1)
        total_correct += torch.sum(preds == labels)
        total_samples += labels.size(0)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        
        if (i + 1) % 50 == 0 or (i + 1) == len(data_loader): 
            elapsed = time.time() - start_time
            current_acc = total_correct.double() / total_samples
            print(f"    [!] Batch {i+1}/{len(data_loader)} | Loss: {loss.item():.4f} | Epoch Accuracy: {current_acc:.4f} | Time: {elapsed:.2f}s")
    
    avg_loss = total_loss / len(data_loader)
    avg_acc = total_correct.double() / total_samples
    return avg_loss, avg_acc

# --- Main Training ---
for epoch in range(EPOCHS):
    print(f"\n[!] --- Epoch {epoch + 1}/{EPOCHS} ---")
    epoch_start_time = time.time()
    
    train_loss, train_acc = train_epoch(
        model,
        train_loader,
        optimizer,
        device,
        scheduler
    )
    
    epoch_time = time.time() - epoch_start_time
    print(f"\n[+] Epoch {epoch + 1} Summary:")
    print(f"  > Train Loss: {train_loss:.4f} | Train Accuracy: {train_acc:.4f}")
    print(f"  > Time: {epoch_time:.2f}s")

In [ ]:
# print("[>] Starting Task: Defining Evaluation Function")

# def eval_model(model, data_loader, device):
#     model.eval() 
    
#     total_loss = 0
#     total_correct = 0
#     total_samples = 0
    
#     all_labels = []
#     all_preds = []
    
#     print("[!] Running evaluation on test set...")
#     start_time = time.time()
    
#     with torch.no_grad():
#         for i, batch in enumerate(data_loader):
#             input_ids = batch['input_ids'].to(device)
#             attention_mask = batch['attention_mask'].to(device)
#             labels = batch['labels'].to(device)
            
#             outputs = model(
#                 input_ids=input_ids,
#                 attention_mask=attention_mask,
#                 labels=labels
#             )
            
#             loss = outputs.loss
#             logits = outputs.logits
            
#             total_loss += loss.item()
#             _, preds = torch.max(logits, dim=1)
#             total_correct += torch.sum(preds == labels)
#             total_samples += labels.size(0)
            
#             all_labels.extend(labels.cpu().numpy())
#             all_preds.extend(preds.cpu().numpy())
            
#             if (i + 1) % 50 == 0 or (i + 1) == len(data_loader):
#                  print(f"  [!] Evaluated Batch {i+1}/{len(data_loader)}")

#     elapsed = time.time() - start_time
#     print(f"[+] Evaluation finished in {elapsed:.2f}s.")
    
#     avg_loss = total_loss / len(data_loader)
#     avg_acc = total_correct.double() / total_samples
#     return avg_loss, avg_acc, all_labels, all_preds

In [ ]:
# print("[>] Starting Task: Report Results (Test Set)")

# # [>] Running final evaluation...
# test_loss, test_acc, y_true, y_pred = eval_model(model, test_loader, device)

# print(f"\n[!] --- Final Model Performance (Test Set) ---")
# print(f"[+]   Test Loss:     {test_loss:.4f}")
# print(f"[+]   Test Accuracy: {test_acc:.4f}")

# # [>] Calculate Precision, Recall, F1 (for the 'Fake' class, label 1)
# precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='binary')

# print(f"[+]   Precision (for Fake):     {precision:.4f}")
# print(f"[+]   Recall (for Fake):        {recall:.4f}")
# print(f"[+]   F1-Score (for Fake):      {f1:.4f}")
# print("[!] ------------------------------------------")

# # [>] Full classification report
# print("\n[+] Classification Report:")
# target_names = ['Non-Fake (0)', 'Fake (1)']
# print(classification_report(y_true, y_pred, target_names=target_names))

# # [>] Confusion Matrix
# cm = confusion_matrix(y_true, y_pred)
# print("\n[+] Confusion Matrix:")
# print(cm)

# plt.figure(figsize=(8, 6))
# sns.heatmap(
#     cm, 
#     annot=True, 
#     fmt='d', 
#     cmap='Blues', 
#     xticklabels=target_names, 
#     yticklabels=target_names
# )
# plt.title('Confusion Matrix - Test Set')
# plt.xlabel('Predicted Label')
# plt.ylabel('True Label')
# plt.show()